# 🗞️ Thai News Classifier
### Multi-Model Classification with Comparison & Visualization
---
This notebook classifies Thai news articles from **Thairath** into categories:
**Entertainment, Lottery, Crime, Disaster, Politics, Automotive, Royal, Education, Economics**

We compare **6 classification models** and visualize their performance with accuracy charts and confusion matrices.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

# 6 Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

print("✅ All libraries imported successfully!")


## 📂 2. Load & Explore Data

In [ ]:
df = pd.read_csv('thairath_test.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## 🏷️ 3. Data Labeling & Augmentation
Since the dataset has no labels, we assign categories based on content analysis. We also generate **synthetic training samples** from category-specific keyword patterns to build a robust classifier.


In [ ]:
# Manual labels based on content/URL analysis
manual_labels = {
    2828481: "Lottery",        # หวย, เลขเด็ด
    2828591: "Crime",          # นักท่องเที่ยวตกโขดหิน
    2828593: "Disaster",       # น้ำท่วมพัทลุง
    2828540: "Lottery",        # เลขเด็ด, หวย
    2828598: "Education",      # TGAT/TPAT สอบ
    2828576: "Lottery",        # เลขเด็ด, หวย
    2828608: "Politics",       # politic URL
    2828480: "Automotive",     # Motor Expo, รถ
    2828612: "Royal",          # สมเด็จพระนางเจ้า
    2828597: "Crime",          # รวบหนุ่มอ้างตัวเป็นหมอ
    2828575: "Lottery",        # เลขเด็ด
    2828604: "Disaster",       # คานถล่ม, จราจร
    2828613: "Politics",       # นายกฯ, ช่วยเหลือน้ำท่วม
    2828603: "Disaster",       # น้ำท่วมยะลา
    2828614: "Entertainment",  # อ๋อม สกาวใจ
    2828618: "Politics",       # ทรัมป์, เอฟบีไอ
    2828616: "Economics",      # ราคาข้าว, มาตรการ
    2828619: "Crime",          # รวบชาย, มีดจี้
    2828621: "Entertainment",  # ย่าเปิ้ล, น้องคากิ
}

df['category'] = df['id'].map(manual_labels)
df['text'] = df['headline'].fillna('') + ' ' + df['content'].fillna('')

print("📊 Category Distribution:")
print(df['category'].value_counts())


In [ ]:
# Thai keyword dictionaries per category — used to generate synthetic training data
category_keywords = {
    "Entertainment": [
        "ดารา คอนเสิร์ต บันเทิง ข่าวดารา เปิดตัว ละคร ซีรีส์",
        "นักร้อง แร็ปเปอร์ ไอดอล ศิลปิน เพลงใหม่ อัลบั้ม",
        "ภาพยนตร์ หนัง รางวัล เทศกาลภาพยนตร์ ฉายรอบแรก",
        "คนดัง ข่าวบันเทิง โซเชียล ไวรัล แฟนคลับ ติดเทรนด์",
        "รายการทีวี วาไรตี้ เกมโชว์ ถ่ายทอดสด พิธีกร รีวิว",
        "แฟชั่น นางแบบ เดินแบบ สัปดาห์แฟชั่น แบรนด์เนม",
        "เปิดตัว คู่รัก แต่งงาน ดารา งานเลี้ยง ปาร์ตี้ ลูกดารา",
        "ดาราโพสต์ อินสตาแกรม ทวิตเตอร์ สตอรี่ แชร์ภาพ โพสต์",
        "นักแสดง พระเอก นางเอก แม่ย่า ลูก แฟนคลับ ข่าวซุบซิบ",
        "ย่าเปิ้ล น้องคากิ แจ็ค แฟนฉัน ใบหม่อน ตัวเลข โชค",
        "อ๋อม สกาวใจ น้องกระดิ่ง ลูกหลาน อวยพร ดวง โชคลาภ",
    ],
    "Lottery": [
        "หวย เลขเด็ด ลอตเตอรี่ สลากกินแบ่ง ออกรางวัล",
        "เลขเด็ดงวดนี้ หวยไทยรัฐ โค้งสุดท้าย ตรวจหวย เสี่ยงโชค",
        "รางวัลที่ 1 เศรษฐี เลขดัง เลขท้าย 2 ตัว ซื้อหวย",
        "แผงลอตเตอรี่ หวยสัญจร ถ่ายทอดสด กองสลาก ขายดี",
        "เลขเด็ดงวดนี้ ให้โชค ถูกรางวัล แนวทาง ตัวเลข บอกใบ้",
        "หวยรัฐบาล งวดประจำวัน สลาก ถูกหวย รางวัลใหญ่ 12 ล้าน",
        "ใบ้เลข เลขดัง คอหวย นำโชค บอกตัวเลข งวดหน้า",
        "ลุ้นหวย ตรวจสลาก เลขท้าย เลขหน้า รางวัลข้างเคียง",
    ],
    "Crime": [
        "จับกุม ตำรวจ ผู้ต้องหา คดี อาชญากรรม สอบสวน",
        "ฆาตกรรม ปล้น ชิงทรัพย์ มีดจี้ ปืน ข่มขืน ยาเสพติด",
        "ศาล พิพากษา จำคุก ประกันตัว หมายจับ จำเลย อัยการ",
        "หลอกลงทุน ฉ้อโกง แก๊งคอลเซ็นเตอร์ โรแมนซ์สแกม หลอกโอน",
        "สืบนครบาล บุกจับ รวบตัว คนร้าย ก่อเหตุ หลบหนี",
        "อุบัติเหตุ รถชน เสียชีวิต บาดเจ็บ พลิกคว่ำ ชนท้าย",
        "มีดจี้ ชิงทรัพย์ จี้ร้านทอง ลักทรัพย์ วิ่งราว ขโมย",
        "รวบหนุ่ม หลอกลงทุน อ้างตัว คลินิก เสียหาย ล้านบาท",
        "ค้ายาบ้า ยาไอซ์ ของกลาง กัญชา ยาอี ยาเค จับยา",
    ],
    "Disaster": [
        "น้ำท่วม พายุ ดินถล่ม ภัยธรรมชาติ อุทกภัย วาตภัย",
        "เตือนภัย ฝนตก น้ำป่า คลื่นซัด ระดับน้ำสูง อพยพ",
        "ภัยพิบัติ แผ่นดินไหว สึนามิ ไฟป่า ภัยแล้ง น้ำท่วมหนัก",
        "กู้ภัย อาสาสมัคร ช่วยเหลือ ผู้ประสบภัย ศูนย์อพยพ ถุงยังชีพ",
        "น้ำท่วมพัทลุง ทะเลสาบสงขลา น้ำเอ่อล้น ระบายน้ำ บ้านจม",
        "น้ำท่วมยะลา นราธิวาส ปัตตานี ภาคใต้ น้ำขัง ถนนจม",
        "คานถล่ม สะพานถล่ม เส้นทางถูกตัด จราจร ทางพิเศษ ทางด่วน",
        "ความเสียหาย รถจม ประกาศเตือน เฝ้าระวัง สถานการณ์ เร่งแก้ไข",
    ],
    "Politics": [
        "การเมือง สภา ผู้แทนราษฎร พรรคการเมือง เลือกตั้ง รัฐสภา",
        "นายกรัฐมนตรี ครม. มติคณะรัฐมนตรี นโยบาย รัฐบาล ฝ่ายค้าน",
        "สมาชิกสภา ส.ส. ส.ว. อภิปราย ลงมติ ญัตติ กฎหมาย",
        "นายกฯ สัญจร ตรวจราชการ เร่งรัด ช่วยเหลือ แพทองธาร",
        "ทรัมป์ ประธานาธิบดี สหรัฐ เอฟบีไอ การเมืองต่างประเทศ",
        "ปฏิรูป แก้ไขรัฐธรรมนูญ อำนาจ กฎหมาย พ.ร.บ. ราชกิจจา",
        "เพื่อไทย ก้าวไกล ประชาธิปัตย์ ภูมิใจไทย พลังประชารัฐ",
        "นโยบายรัฐ มาตรการ ประกาศ สั่งการ บริหาร ราชการ แต่งตั้ง",
    ],
    "Automotive": [
        "รถยนต์ มอเตอร์โชว์ Motor Expo รถใหม่ เปิดตัวรถ",
        "รถไฟฟ้า EV ยานยนต์ ชาร์จ แบตเตอรี่ พลังงานสะอาด",
        "รถกระบะ รถเก๋ง SUV ซูเปอร์คาร์ สปอร์ต มอเตอร์ไซค์",
        "ทดสอบรถ รีวิวรถ สเปก ราคา โปรโมชั่น ผ่อน ดาวน์",
        "โตโยต้า ฮอนด้า นิสสัน เบนซ์ บีเอ็ม เทสลา BYD",
        "งานมอเตอร์โชว์ อิมแพ็ค เมืองทอง รถน่าสนใจ รถเด่น",
    ],
    "Royal": [
        "ราชวงศ์ พระราชพิธี สมเด็จพระ พระบรมราชินี เสด็จ",
        "พระเจ้าอยู่หัว พระราชดำรัส พระราชกรณียกิจ ในหลวง",
        "สวนสนาม ทหาร พิธีสวนสนาม ถวายสัตย์ปฏิญาณ กองทัพ",
        "พระราชทาน ปริญญา พระราชพิธี พระมหากษัตริย์ จงรักภักดี",
    ],
    "Education": [
        "สอบ มหาวิทยาลัย นักเรียน นักศึกษา ทุนการศึกษา TCAS",
        "TGAT TPAT สอบเข้า เลื่อนสอบ สนามสอบ คะแนนสอบ",
        "โรงเรียน ครู อาจารย์ หลักสูตร การศึกษา กศน. สพฐ.",
        "ปริญญา จบการศึกษา รับสมัคร เปิดเทอม ปิดเทอม สอบปลายภาค",
    ],
    "Economics": [
        "เศรษฐกิจ GDP อัตราแลกเปลี่ยน เงินเฟ้อ ดอกเบี้ย ธนาคาร",
        "ราคาข้าว สินค้าเกษตร ราคาทอง ตลาดหุ้น หุ้นไทย SET",
        "ส่งออก นำเข้า การค้า การลงทุน FDI งบประมาณ รายได้",
        "ครม.เคาะ มาตรการเศรษฐกิจ พาณิชย์ อุตสาหกรรม เงินบาท",
        "รักษาเสถียรภาพ ราคาข้าว ข้าวโพด เกษตรกร สินเชื่อ ธ.ก.ส.",
        "มาตรการรักษาเสถียรภาพ ราคาสินค้าเกษตร กระทรวงพาณิชย์ รมว.",
        "ค่าแรง ค่าครองชีพ สวัสดิการ เงินอุดหนุน ดิจิทัลวอลเล็ต",
    ],
}

# Generate synthetic training data
synthetic_data = []
for category, phrases_list in category_keywords.items():
    for phrases in phrases_list:
        synthetic_data.append({'text': phrases, 'category': category})
    for i in range(len(phrases_list)):
        for j in range(i+1, min(i+3, len(phrases_list))):
            combined = phrases_list[i] + " " + phrases_list[j]
            synthetic_data.append({'text': combined, 'category': category})

synthetic_df = pd.DataFrame(synthetic_data)
train_df = pd.concat([df[['text', 'category']], synthetic_df], ignore_index=True)

print(f"📊 Training dataset size: {len(train_df)}")
print(f"\nCategory distribution in training data:")
print(train_df['category'].value_counts())


## 🧹 4. Text Preprocessing

In [ ]:
def preprocess_thai_text(text):
    """Clean Thai text for classification."""
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\u0E00-\u0E7Fa-zA-Z0-9\s]', ' ', text)
    return text

train_df['text_clean'] = train_df['text'].apply(preprocess_thai_text)
df['text_clean'] = df['text'].apply(preprocess_thai_text)

print("✅ Text preprocessing complete!")
print(f"\nSample cleaned text:")
print(train_df['text_clean'].iloc[0][:120], "...")


## ⚙️ 5. Feature Extraction & Train/Test Split

In [ ]:
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['category'])

X = train_df['text_clean']
y = train_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# TF-IDF with character n-grams (works great for Thai without word segmentation)
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=10000, sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"TF-IDF features: {X_train_tfidf.shape[1]}")
print(f"Categories: {list(le.classes_)}")


## 🏋️ 6. Train & Evaluate 6 Models

In [ ]:
models = {
    '1. Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    '2. Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    '3. Linear SVM (SVC)': LinearSVC(max_iter=2000, C=1.0, random_state=42),
    '4. Random Forest': RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42),
    '5. Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=42),
    '6. K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3, metric='cosine'),
}

results = {}
predictions = {}

print("=" * 70)
print("🏋️ TRAINING 6 MODELS...")
print("=" * 70)

for name, model in models.items():
    print(f"\n{'─' * 50}")
    print(f"📌 {name}")
    print(f"{'─' * 50}")
    
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    predictions[name] = y_pred
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring='accuracy')
    
    results[name] = {
        'accuracy': acc, 'f1_score': f1,
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
        'model': model,
    }
    
    print(f"  Test Accuracy : {acc:.4f}")
    print(f"  F1 Score      : {f1:.4f}")
    print(f"  CV Accuracy   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


## 📊 7. Model Comparison Table

In [ ]:
results_df = pd.DataFrame({
    name: {
        'Test Accuracy': f"{v['accuracy']:.4f}",
        'F1 Score (weighted)': f"{v['f1_score']:.4f}",
        'CV Accuracy (mean)': f"{v['cv_mean']:.4f}",
        'CV Std': f"{v['cv_std']:.4f}",
    }
    for name, v in results.items()
}).T

print("\n" + "=" * 70)
print("📊 MODEL COMPARISON TABLE")
print("=" * 70)
results_df


## 📈 8. Visualization — Accuracy & F1 Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

model_names = [name.split('. ')[1] for name in results.keys()]
accuracies = [v['accuracy'] for v in results.values()]
f1_scores_list = [v['f1_score'] for v in results.values()]
x = np.arange(len(model_names))
width = 0.35

bars1 = axes[0].bar(x - width/2, accuracies, width, label='Test Accuracy', color='#2196F3', alpha=0.85)
bars2 = axes[0].bar(x + width/2, f1_scores_list, width, label='F1 Score', color='#FF9800', alpha=0.85)
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('🏆 Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.15)
axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.3)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

cv_means = [v['cv_mean'] for v in results.values()]
cv_stds = [v['cv_std'] for v in results.values()]
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800', '#00BCD4']
bars = axes[1].bar(model_names, cv_means, yerr=cv_stds, capsize=5,
                    color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('CV Accuracy', fontsize=12)
axes[1].set_title('📈 Cross-Validation Accuracy (± Std)', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.15)
for bar, mean in zip(bars, cv_means):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.04,
                 f'{mean:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 🔍 9. Confusion Matrices — All 6 Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes_flat = axes.flatten()
category_names = le.classes_

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=category_names, yticklabels=category_names,
                ax=axes_flat[idx], cbar=False, linewidths=0.5, annot_kws={"size": 10})
    acc = results[name]['accuracy']
    short_name = name.split('. ')[1]
    axes_flat[idx].set_title(f'{short_name}\n(Accuracy: {acc:.2%})', fontsize=12, fontweight='bold')
    axes_flat[idx].set_xlabel('Predicted', fontsize=10)
    axes_flat[idx].set_ylabel('Actual', fontsize=10)
    axes_flat[idx].tick_params(axis='both', labelsize=8)
    plt.setp(axes_flat[idx].get_xticklabels(), rotation=45, ha='right')

plt.suptitle('🔍 Confusion Matrices — All 6 Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🏆 10. Best Model & Classification Report

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_result = results[best_model_name]

print("=" * 70)
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Test Accuracy : {best_result['accuracy']:.4f}")
print(f"   F1 Score      : {best_result['f1_score']:.4f}")
print(f"   CV Accuracy   : {best_result['cv_mean']:.4f} ± {best_result['cv_std']:.4f}")
print("=" * 70)

print("\n📋 Detailed Classification Report (Best Model):")
print(classification_report(y_test, predictions[best_model_name], target_names=le.classes_))


## 🧪 11. Test on Original 18 Articles

In [ ]:
best_model = best_result['model']
X_original_tfidf = tfidf.transform(df['text_clean'])
df['predicted'] = le.inverse_transform(best_model.predict(X_original_tfidf))

for _, row in df.iterrows():
    match = "✅" if row['category'] == row['predicted'] else "❌"
    print(f"{match} [{row['predicted']:>15}] (actual: {row['category']:>15}) | {row['headline'][:60]}")

original_acc = accuracy_score(le.transform(df['category']), best_model.predict(X_original_tfidf))
print(f"\n📊 Accuracy on original articles: {original_acc:.2%}")


## 🎯 12. Interactive Prediction — Classify Your Own Text!
The `classify_news()` function runs all 6 models and uses **majority voting** for the final prediction.


In [ ]:
def classify_news(text, show_all_models=True):
    """Classify a Thai news text into a category using all 6 models with majority vote."""
    cleaned = preprocess_thai_text(text)
    text_tfidf = tfidf.transform([cleaned])
    
    print(f"\n{'=' * 60}")
    print(f"📰 Input: {text[:80]}...")
    print(f"{'=' * 60}")
    
    if show_all_models:
        print("\n🔮 Predictions from all models:")
        print(f"{'─' * 45}")
        
        votes = {}
        for name, res in results.items():
            model = res['model']
            pred_label = le.inverse_transform(model.predict(text_tfidf))[0]
            short_name = name.split('. ')[1]
            votes[pred_label] = votes.get(pred_label, 0) + 1
            print(f"  {short_name:.<30} {pred_label}")
        
        majority = max(votes, key=votes.get)
        print(f"\n{'─' * 45}")
        print(f"🏆 Majority Vote ({votes[majority]}/6): ➜  {majority}")
        return majority
    else:
        pred = le.inverse_transform(best_model.predict(text_tfidf))[0]
        print(f"\n🏆 Predicted Category: {pred}")
        return pred

# --- Example predictions ---
classify_news("ธนาคารแห่งประเทศไทยปรับอัตราดอกเบี้ยนโยบาย ส่งผลต่อตลาดการเงินและเศรษฐกิจไทย")
classify_news("ลิซ่า BLACKPINK คว้ารางวัล Best K-Pop ที่งาน MTV VMAs สร้างประวัติศาสตร์ศิลปินไทย")
classify_news("สภาผู้แทนราษฎร ลงมติผ่านร่างพระราชบัญญัติงบประมาณรายจ่ายประจำปี 2568 วาระแรก")
classify_news("น้ำท่วมหนักภาคใต้ สงขลา ยะลา นราธิวาส ปัตตานี ฝนตกหนัก ประชาชนเดือดร้อน")
classify_news("ตำรวจจับกุมแก๊งคอลเซ็นเตอร์ หลอกโอนเงิน เสียหายกว่า 50 ล้านบาท")


## 💬 13. User Input — Type Your Own Text!
**Run the cell below, type any Thai news headline, and see the classification result!**


In [ ]:
# === USER INPUT SECTION ===
# Change the text below to classify any Thai news!

user_text = "ราคาน้ำมันปรับขึ้น 2 บาท ส่งผลค่าครองชีพสูง ประชาชนรับผลกระทบ"

result = classify_news(user_text)
print(f"\n✅ Final Category: {result}")


In [ ]:
# === INTERACTIVE MODE (uncomment to use) ===
# Run this cell and type your text in the input box

# while True:
#     user_input = input("\n📝 Enter Thai news text (or 'quit' to exit): ")
#     if user_input.lower() in ['quit', 'exit', 'q']:
#         print("👋 Goodbye!")
#         break
#     if user_input.strip():
#         classify_news(user_input)
#     else:
#         print("⚠️ Please enter some text.")


---
## 📝 Summary

| Item | Detail |
|------|--------|
| Dataset | 18 Thairath news articles + synthetic augmentation |
| Categories | 9: Automotive, Crime, Disaster, Economics, Education, Entertainment, Lottery, Politics, Royal |
| Models | 6: Logistic Regression, Naive Bayes, Linear SVM, Random Forest, Gradient Boosting, KNN |
| Best Method | Majority voting across all 6 models |
| Features | TF-IDF character n-grams (2-5) — works well for Thai without word segmentation |
